In [1]:
import torch
import triton
import triton.language as tl

DEVICE = triton.runtime.driver.active.get_active_torch_device()

In [2]:
# the kernel
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # setup
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    # load
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    # compute
    out = x + y

    # store
    tl.store(out_ptr + offsets, out, mask=mask)

In [5]:
# add helper
def add(x: torch.Tensor, y: torch.Tensor):
    out = torch.zeros_like(x)
    n_elements = out.numel()

    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    add_kernel[grid](x, y, out, n_elements, BLOCK_SIZE=1024)

    return out

In [6]:
torch.manual_seed(0)
size = 98432
x = torch.rand(size, device=DEVICE)
y = torch.rand(size, device=DEVICE)
output_torch = x + y
output_triton = add(x, y)
print(output_torch)
print(output_triton)

tensor([1.3713, 1.3076, 0.4940,  ..., 1.1147, 1.1906, 1.5746], device='cuda:0')
tensor([1.3713, 1.3076, 0.4940,  ..., 1.1147, 1.1906, 1.5746], device='cuda:0')
